# Treinamento e avaliação do modelo

Objetivo: usar os sete fechamentos diários mais recentes do Bitcoin para estimar o fechamento do dia seguinte. A previsão será feita depois que o fechamento do dia atual estiver disponível.

O notebook prepara as entradas e o alvo, separa treino e teste, treina uma regressão linear, avalia o erro e salva o modelo em `modelos/modelo.joblib`. Execute as células de cima para baixo, a partir da pasta do projeto, após gerar `dados/bitcoin_5_anos.csv`.

## Dependências

As versões usadas neste treinamento estão em `requirements-treinamento.txt`. A primeira célula instala essas versões no ambiente Python do notebook. Elas também ficam registradas junto ao modelo para reproduzir o ambiente de inferência.

Referência: [persistência de modelos no scikit-learn](https://scikit-learn.org/stable/model_persistence.html).

In [1]:
%pip install -q -r requirements-treinamento.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /home/kaian/Documentos/Murilao/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Ler e conferir o histórico

Vamos usar `Date` e `Close`. A conferência das datas garante que cada linha represente o dia seguinte à anterior: assim, deslocar uma linha equivale a deslocar um dia.

In [2]:
from pathlib import Path

import pandas as pd

arquivo_csv = Path('dados/bitcoin_5_anos.csv')
janela = 7

dados = pd.read_csv(arquivo_csv, parse_dates=['Date'])
dados = dados[['Date', 'Close']].sort_values('Date').reset_index(drop=True)

if len(dados) < janela + 2:
    raise ValueError('O histórico é insuficiente para preparar treino e teste.')
if dados.isna().any().any():
    raise ValueError('Há datas ou fechamentos ausentes. Confira o CSV antes de continuar.')
if dados['Date'].duplicated().any():
    raise ValueError('Há datas duplicadas no CSV. Confira a coleta.')

datas_esperadas = pd.date_range(dados['Date'].min(), dados['Date'].max(), freq='D')
if not pd.DatetimeIndex(dados['Date']).equals(datas_esperadas):
    raise ValueError('Faltam dias no histórico. Confira a coleta antes de usar os deslocamentos.')

print(f'Registros no histórico: {len(dados)}')
dados.head()

Registros no histórico: 1826


,Date,Close
0,2021-10-05,51514.812500
1,2021-10-06,55361.449219
2,2021-10-07,53805.984375
3,2021-10-08,53967.847656
4,2021-10-09,54968.222656


## 2. Montar as entradas e o alvo

Cada amostra representa uma previsão feita ao final de um dia. `fechamento_dia_0` é o fechamento desse dia; `fechamento_dia_1` é o do dia anterior, até `fechamento_dia_6`. Essas sete colunas serão as entradas do modelo.

`shift(1)` traz o valor da linha anterior; `shift(6)` traz o de seis dias antes. `shift(-1)` traz o fechamento do dia seguinte, que será o alvo que o modelo tentará aprender a prever. O alvo fica separado das entradas.

As seis primeiras linhas ainda não têm sete fechamentos disponíveis. A última ainda não tem fechamento do dia seguinte. Essas linhas são removidas da base de amostras.

In [3]:
base = dados[['Date']].copy()
colunas_entrada = []

for atraso in range(janela):
    nome_coluna = f'fechamento_dia_{atraso}'
    base[nome_coluna] = dados['Close'].shift(atraso)
    colunas_entrada.append(nome_coluna)

base['data_previsao'] = dados['Date'].shift(-1)
base['fechamento_seguinte'] = dados['Close'].shift(-1)

linhas_antes = len(base)
base = base.dropna().reset_index(drop=True)

print(f'Amostras válidas: {len(base)}')
print(f'Linhas removidas por falta de histórico ou alvo: {linhas_antes - len(base)}')
base.head(3)

Amostras válidas: 1819
Linhas removidas por falta de histórico ou alvo: 7


,Date,fechamento_dia_0,fechamento_dia_1,fechamento_dia_2,fechamento_dia_3,fechamento_dia_4,fechamento_dia_5,fechamento_dia_6,data_previsao,fechamento_seguinte
0,2021-10-11,57484.789062,54771.578125,54968.222656,53967.847656,53805.984375,55361.449219,51514.812500,2021-10-12,56041.058594
1,2021-10-12,56041.058594,57484.789062,54771.578125,54968.222656,53967.847656,53805.984375,55361.449219,2021-10-13,57401.097656
2,2021-10-13,57401.097656,56041.058594,57484.789062,54771.578125,54968.222656,53967.847656,53805.984375,2021-10-14,57321.523438


## 3. Separar treino e teste em ordem cronológica

Os primeiros 80% das amostras serão usados no treino; os 20% mais recentes, no teste. As linhas não são embaralhadas.

O teste vai simular previsões do dia seguinte usando fechamentos já observados até cada dia. Por isso, uma amostra de teste pode usar dias anteriores do período de treino como histórico. Essa avaliação é de um dia à frente, e não uma previsão de todo o período de teste de uma só vez.

In [4]:
proporcao_treino = 0.8
ponto_corte = int(len(base) * proporcao_treino)

treino = base.iloc[:ponto_corte].copy()
teste = base.iloc[ponto_corte:].copy()

print(f'Treino: {len(treino)} amostras')
print(f'Teste: {len(teste)} amostras')
print(f'Datas previstas no treino: {treino["data_previsao"].min().date()} a {treino["data_previsao"].max().date()}')
print(f'Datas previstas no teste: {teste["data_previsao"].min().date()} a {teste["data_previsao"].max().date()}')

Treino: 1455 amostras
Teste: 364 amostras
Datas previstas no treino: 2021-10-12 a 2025-10-05
Datas previstas no teste: 2025-10-06 a 2026-10-04


## 4. Separar as entradas da resposta esperada

`X` contém os sete fechamentos usados como entradas. `y` contém o fechamento do dia seguinte, que é a resposta esperada. As datas servem para acompanhar os períodos e ficam fora de `X`.

In [5]:
X_treino = treino[colunas_entrada].copy()
y_treino = treino['fechamento_seguinte'].copy()

X_teste = teste[colunas_entrada].copy()
y_teste = teste['fechamento_seguinte'].copy()

print(f'Entradas de treino: {X_treino.shape[0]} amostras e {X_treino.shape[1]} colunas')
print(f'Alvos de treino: {len(y_treino)}')
print(f'Entradas de teste: {X_teste.shape[0]} amostras e {X_teste.shape[1]} colunas')
print(f'Alvos de teste: {len(y_teste)}')

X_treino.head(3)

Entradas de treino: 1455 amostras e 7 colunas
Alvos de treino: 1455
Entradas de teste: 364 amostras e 7 colunas
Alvos de teste: 364


,fechamento_dia_0,fechamento_dia_1,fechamento_dia_2,fechamento_dia_3,fechamento_dia_4,fechamento_dia_5,fechamento_dia_6
0,57484.789062,54771.578125,54968.222656,53967.847656,53805.984375,55361.449219,51514.812500
1,56041.058594,57484.789062,54771.578125,54968.222656,53967.847656,53805.984375,55361.449219
2,57401.097656,56041.058594,57484.789062,54771.578125,54968.222656,53967.847656,53805.984375


## 5. Treinar uma regressão linear

A regressão linear aprende uma combinação dos sete fechamentos para estimar o próximo. Foi escolhida como uma primeira abordagem simples para demonstrar o treinamento e a integração com o backend.

`fit(X_treino, y_treino)` ajusta o modelo usando somente o conjunto de treino. Os preços do conjunto de teste serão usados para avaliar as previsões.

Referência: [LinearRegression no scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [6]:
import json
import sys

import joblib
import numpy as np
import scipy
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

print(f'Modelo: {type(modelo).__name__}')
print(f'Amostras usadas no treinamento: {len(X_treino)}')
print(f'Entradas por amostra: {modelo.n_features_in_}')
print(f'Versão do scikit-learn: {sklearn.__version__}')

Modelo: LinearRegression
Amostras usadas no treinamento: 1455
Entradas por amostra: 7
Versão do scikit-learn: 1.9.0


## 6. Avaliar as previsões

Vamos comparar as previsões com os fechamentos reais das 364 amostras de teste:

- **MAE:** média do erro absoluto, em dólares. Indica quanto a previsão se afastou do valor real, em média.
- **RMSE:** também está em dólares, mas dá mais peso aos erros grandes.

Quanto menores esses valores, menor o erro. Eles não representam uma porcentagem de acerto.

Como referência, usamos uma previsão simples: o fechamento de amanhã será igual ao último fechamento conhecido. Isso permite verificar se o modelo melhorou em relação a repetir esse preço.

Referência: [métricas de regressão no scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).

In [7]:
predicoes = modelo.predict(X_teste)
predicoes_referencia = X_teste['fechamento_dia_0'].to_numpy()

metricas = pd.DataFrame([
    {
        'metodo': 'Regressão linear',
        'MAE_USD': float(mean_absolute_error(y_teste, predicoes)),
        'RMSE_USD': float(root_mean_squared_error(y_teste, predicoes)),
    },
    {
        'metodo': 'Repetir último fechamento',
        'MAE_USD': float(mean_absolute_error(y_teste, predicoes_referencia)),
        'RMSE_USD': float(root_mean_squared_error(y_teste, predicoes_referencia)),
    },
])

print(metricas.to_string(index=False, float_format=lambda valor: f'{valor:.2f}'))

mae_modelo = metricas.loc[0, 'MAE_USD']
mae_referencia = metricas.loc[1, 'MAE_USD']
if mae_modelo < mae_referencia:
    print('Neste teste, a regressão linear teve MAE menor que a referência.')
else:
    print('Neste teste, a regressão linear não teve MAE menor que a referência.')

                   metodo  MAE_USD  RMSE_USD
         Regressão linear  1317.22   1885.91
Repetir último fechamento  1303.33   1867.64
Neste teste, a regressão linear não teve MAE menor que a referência.


In [8]:
comparacao = pd.DataFrame({
    'data_prevista': teste['data_previsao'].to_numpy(),
    'fechamento_real_USD': y_teste.to_numpy(),
    'previsao_modelo_USD': predicoes,
    'previsao_referencia_USD': predicoes_referencia,
})
comparacao['erro_absoluto_modelo_USD'] = (
    comparacao['fechamento_real_USD'] - comparacao['previsao_modelo_USD']
).abs()

comparacao.head()

,data_prevista,fechamento_real_USD,previsao_modelo_USD,previsao_referencia_USD,erro_absoluto_modelo_USD
0,2025-10-06,124752.531250,123584.733134,123513.476562,1167.798116
1,2025-10-07,121451.382812,124602.908135,124752.531250,3151.525322
2,2025-10-08,123354.867188,121662.735737,121451.382812,1692.131451
3,2025-10-09,121705.585938,123283.106731,123354.867188,1577.520794
4,2025-10-10,113214.367188,121954.767850,121705.585938,8740.400663


## 7. Salvar o modelo e os resultados

`joblib.dump` salva o modelo treinado em `modelos/modelo.joblib`, o arquivo previsto no UML. O backend poderá carregá-lo sem repetir o treinamento.

`modelos/metadados.json` registra a ordem das entradas, os períodos, as métricas e as versões das bibliotecas. A ordem é do fechamento mais recente até o de seis dias antes.

O modelo exportado é o mesmo que foi avaliado: seu ajuste usou somente o conjunto de treino.

In [9]:
pasta_modelos = Path('modelos')
pasta_modelos.mkdir(parents=True, exist_ok=True)
arquivo_modelo = pasta_modelos / 'modelo.joblib'
arquivo_metadados = pasta_modelos / 'metadados.json'

joblib.dump(modelo, arquivo_modelo)

metadados = {
    'modelo': type(modelo).__name__,
    'moeda': 'BTC-USD',
    'fonte_csv': str(arquivo_csv),
    'frequencia': 'diaria',
    'janela_dias': janela,
    'horizonte_dias': 1,
    'colunas_entrada': colunas_entrada,
    'ordem_entradas': 'do fechamento mais recente ao de seis dias antes',
    'amostras_treino': len(treino),
    'amostras_teste': len(teste),
    'periodo_alvos_treino': {
        'inicio': treino['data_previsao'].min().date().isoformat(),
        'fim': treino['data_previsao'].max().date().isoformat(),
    },
    'periodo_alvos_teste': {
        'inicio': teste['data_previsao'].min().date().isoformat(),
        'fim': teste['data_previsao'].max().date().isoformat(),
    },
    'versoes': {
        'python': sys.version.split()[0],
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'scipy': scipy.__version__,
        'scikit-learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
    'metricas_teste': metricas.to_dict(orient='records'),
}
arquivo_metadados.write_text(
    json.dumps(metadados, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print(f'Modelo salvo em: {arquivo_modelo}')
print(f'Metadados salvos em: {arquivo_metadados}')

Modelo salvo em: modelos/modelo.joblib
Metadados salvos em: modelos/metadados.json


## 8. Carregar o arquivo e conferir uma predição

A última etapa carrega o modelo salvo, verifica se ele mantém as mesmas previsões e mostra um exemplo do conjunto de teste com seu fechamento real. Essa verificação confirma que o artefato pode ser usado após o treinamento.

In [10]:
modelo_carregado = joblib.load(arquivo_modelo)
predicoes_carregado = modelo_carregado.predict(X_teste)

if not np.allclose(predicoes, predicoes_carregado):
    raise RuntimeError('As previsões mudaram após carregar o modelo salvo.')

entrada_exemplo = X_teste.iloc[[0]]
previsao_exemplo = float(modelo_carregado.predict(entrada_exemplo)[0])
fechamento_real = float(y_teste.iloc[0])
data_exemplo = teste['data_previsao'].iloc[0].date()

print('Modelo carregado: previsões conferem com as do modelo em memória.')
print(f'Data prevista no exemplo de teste: {data_exemplo}')
print(f'Fechamentos de entrada: {entrada_exemplo.iloc[0].tolist()}')
print(f'Previsão: USD {previsao_exemplo:.2f}')
print(f'Fechamento real: USD {fechamento_real:.2f}')
print(f'Erro absoluto: USD {abs(previsao_exemplo - fechamento_real):.2f}')

Modelo carregado: previsões conferem com as do modelo em memória.
Data prevista no exemplo de teste: 2025-10-06
Fechamentos de entrada: [123513.4765625, 122425.4296875, 122266.53125, 120681.2578125, 118648.9296875, 114056.0859375, 114400.3828125]
Previsão: USD 123584.73
Fechamento real: USD 124752.53
Erro absoluto: USD 1167.80


## Limitações e integração realizada

A solução prevê um dia à frente usando somente preços de fechamento. Não considera notícias, volume ou outros fatores. Uma janela de sete dias e um único modelo não garantem precisão. Os resultados são experimentais e não devem ser tratados como recomendação de investimento.

O backend Python foi implementado em `backend/main.py` e executado com Docker Compose. A pasta `modelos/` é montada como volume somente leitura. O serviço carrega o artefato exportado e disponibiliza `POST /predict` e `GET /health`.

Os comandos, justificativas, dificuldades e resultados estão no `readme.md`. A verificação HTTP pode ser repetida com `python scripts/verificar_backend.py`, depois de iniciar o container e instalar as dependências locais.